In [36]:
import pandas as pd
import numpy as np

In [37]:
rfm = pd.read_csv('../data/processed/rfm_features.csv')

In [38]:
rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,2,0.00
1,12347.0,2,7,4310.00
2,12348.0,75,4,1437.24
3,12349.0,19,1,1457.55
4,12350.0,310,1,294.40


In [39]:
!pip install lifetimes --quiet

In [40]:
from lifetimes.utils import summary_data_from_transaction_data

transactions = pd.read_csv('../data/processed/cleaned_transactions.csv')


In [41]:
from lifetimes.utils import summary_data_from_transaction_data

summary = summary_data_from_transaction_data(
    transactions,
    customer_id_col='CustomerID',
    datetime_col='InvoiceDate',
    monetary_value_col='TotalAmount',
    observation_period_end='2011-12-09'
)

summary.head()

,frequency,recency,T,monetary_value
CustomerID,,,,
12346.0,0.0,0.0,325.0,0.000000
12347.0,6.0,365.0,367.0,599.701667
12348.0,3.0,283.0,358.0,261.480000
12349.0,0.0,0.0,18.0,0.000000
12350.0,0.0,0.0,310.0,0.000000


In [42]:
from lifetimes import BetaGeoFitter

bgf = BetaGeoFitter(penalizer_coef=0.0)

bgf.fit(
    summary['frequency'],
    summary['recency'],
    summary['T']
)

bgf.summary



,coef,se(coef),lower 95% bound,upper 95% bound
r,0.844042,0.026370,0.792356,0.895727
alpha,56.467675,2.125597,52.301504,60.633845
a,0.020165,0.006220,0.007974,0.032356
b,0.477809,0.179910,0.125185,0.830432


In [43]:
summary['probability_alive'] = bgf.conditional_probability_alive(
    summary['frequency'],
    summary['recency'],
    summary['T']
)

summary['predicted_purchases_90d'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    90,
    summary['frequency'],
    summary['recency'],
    summary['T']
)

summary.loc[[12346, 12347]]

c:\ProgramData\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,frequency,recency,T,monetary_value,probability_alive,predicted_purchases_90d
CustomerID,,,,,,
12346.0,0.0,0.0,325.0,0.000000,1.000000,0.197608
12347.0,6.0,365.0,367.0,599.701667,0.996212,1.445597


In [44]:
returning_customers = summary[summary['frequency'] > 0]

returning_customers[['frequency', 'monetary_value']].corr()

,frequency,monetary_value
frequency,1.000000,0.190732
monetary_value,0.190732,1.000000


In [45]:
returning_customers = summary[(summary['frequency'] > 0) & (summary['monetary_value'] > 0)]

In [46]:
from lifetimes import GammaGammaFitter

ggf = GammaGammaFitter(penalizer_coef=0.0)

ggf.fit(
    returning_customers['frequency'],
    returning_customers['monetary_value'],
)

ggf.summary

,coef,se(coef),lower 95% bound,upper 95% bound
p,1.670245,0.091724,1.490466,1.850024
q,3.495837,0.152582,3.196777,3.794898
v,511.416504,48.381418,416.588925,606.244083


In [47]:
summary_returning = summary[summary['frequency'] > 0].copy()

summary_returning['predicted_clv'] = ggf.customer_lifetime_value(
    bgf,
    summary_returning['frequency'],
    summary_returning['recency'],
    summary_returning['T'],
    summary_returning['monetary_value'],
    time=12,           # months to project forward
    freq='D',          # your T/recency are in days
    discount_rate=0.01 # standard monthly discount rate assumption
)

summary_returning.loc[[12347]]

,frequency,recency,T,monetary_value,probability_alive,predicted_purchases_90d,predicted_clv
CustomerID,,,,,,,
12347.0,6.0,365.0,367.0,599.701667,0.996212,1.445597,2957.89978


In [48]:
one_time_buyers = summary[summary['frequency'] == 0].copy()

In [49]:
rfm_lookup = rfm.set_index('CustomerID')['Monetary']

one_time_buyers['predicted_clv'] = one_time_buyers.index.map(rfm_lookup)

In [50]:
final_clv = pd.concat([
    summary_returning[['frequency', 'recency', 'T', 'monetary_value', 'probability_alive', 'predicted_clv']],
    one_time_buyers[['frequency', 'recency', 'T', 'monetary_value', 'predicted_clv']]
])

final_clv['probability_alive'] = final_clv['probability_alive'].fillna(1.0)

final_clv.shape

(4362, 6)

In [ ]:
final_clv.to_csv('../data/processed/clv_data.csv')
final_clv.head()


,frequency,recency,T,monetary_value,probability_alive,predicted_clv
CustomerID,,,,,,
12347.0,6.0,365.0,367.0,599.701667,0.996212,2957.899780
12348.0,3.0,283.0,358.0,261.480000,0.982772,879.497234
12352.0,6.0,260.0,296.0,161.485000,0.992363,1273.603054
12356.0,2.0,303.0,325.0,269.905000,0.984100,737.574179
12358.0,1.0,149.0,150.0,523.200000,0.959156,1177.330656
